# BERT++ — BERT-Large-Scale Pretraining on Constrained Hardware

PaLM-style parallel blocks, SwiGLU FFNs, XPos rotary embeddings, a tied-weight MLM head,
and an FSDP training path with activation checkpointing and mixed precision.

Model and data code lives in **`../src/setup_bertpp.py`**; the distributed entry point is
**`../src/train.py`**. This notebook imports that code and smoke-tests it, so what runs here
is exactly what `torchrun` trains.


**Setup**


In [ ]:
# Colab / fresh environment: install dependencies (see ../requirements.txt for pins)
# !pip install -q torch transformers datasets wandb

import sys, pathlib
sys.path.insert(0, str(pathlib.Path("..") / "src"))   # notebook lives in notebooks/

import torch
from setup_bertpp import (
    get_tokenizer, get_datasets, get_dataloader, make_mlm_collate,
    TransformerModel, count_parameters,
)

tokenizer = get_tokenizer()
print(f"vocab: {tokenizer.vocab_size} | mask token id: {tokenizer.mask_token_id}")


**Preprocessing**

The MLM collator implements BERT's 15% / 80-10-10 masking scheme by hand. Vocab size and
special-token ids are captured from the tokenizer, and special tokens are never selected
for masking.

The demo below runs **offline** on four hardcoded sentences, so the collator's behavior is
verifiable without streaming a byte of C4. Real training streams The Pile + C4 via
`get_datasets` — see the training section.


In [ ]:
texts = [
    "The quick brown fox jumps over the lazy dog.",
    "Kubernetes schedules containers across a cluster of worker nodes.",
    "Masked language modeling predicts held-out tokens from context.",
    "Short one.",
]
samples = [tokenizer(t, truncation=True, max_length=512) for t in texts]

collate = make_mlm_collate(tokenizer)
batch = collate(samples)

for k, v in batch.items():
    print(f"{k:>15}: shape {tuple(v.shape)}")

masked = (batch["labels"] != -100)
real   = batch["attention_mask"].bool()
print(f"\nmasked positions: {int(masked.sum())} of {int(real.sum())} real tokens "
      f"({100*masked.sum()/real.sum():.1f}% — target ~15%)")

# invariants
specials = torch.tensor(tokenizer.all_special_ids)
orig_ids = torch.full_like(batch['input_ids'], -1)
for i, s in enumerate(samples):
    orig_ids[i, :len(s['input_ids'])] = torch.tensor(s['input_ids'])
assert not masked[~real].any(),                     "padding is never a label"
assert not torch.isin(orig_ids[masked], specials).any(), "[CLS]/[SEP] never masked"
assert (batch['labels'][masked] == orig_ids[masked]).all(), "labels hold ORIGINAL ids"
print("collator invariants hold ✓")


**Architecture**

One model class at BERT-Large depth and width: **24 layers × 16 heads × 1024 dim**.
The SwiGLU FFN follows the ~(8/3)·d sizing convention (2816 for d=1024), which keeps
parameters comparable to a vanilla 4·d FFN — the tied model is **≈340.9M parameters**
(340,925,242).

One attention detail worth knowing: SDPA's boolean-mask convention is *True = may attend*.
The padded smoke test below exercises the masked path directly.


In [ ]:
model = TransformerModel(vocab_size=tokenizer.vocab_size)   # defaults = BERT-Large scale
n = count_parameters(model)
print(f"parameters (tied weights counted once): {n:,}  (~{n/1e6:.1f}M)")

# CPU smoke test: forward + loss on a padded batch — runs in seconds, no GPU needed
model.eval()
with torch.no_grad():
    smoke = collate(samples)                      # uneven lengths -> real padding
    logits = model(smoke["input_ids"], attention_mask=smoke["attention_mask"])
    print("logits:", tuple(logits.shape))
    loss, _ = model(smoke["input_ids"],
                    attention_mask=smoke["attention_mask"],
                    labels=smoke["labels"])
    print(f"MLM loss on random init: {loss.item():.3f} "
          f"(≈ ln(vocab) = {torch.log(torch.tensor(float(tokenizer.vocab_size))):.3f} — sanity ✓)")


**Training (`src/train.py`)**

The real run is a script, not a notebook cell — launch it with `torchrun`:

```bash
export WANDB_API_KEY=...       # or `wandb login` once
torchrun --nproc_per_node=3 src/train.py
```

What the entry point does:
- FSDP with a `transformer_auto_wrap_policy` over the block class and explicit `FULL_SHARD` —
  the per-layer gather/release cycle is where the memory savings come from;
- bf16 autocast where supported, fp16 + GradScaler otherwise (`torch.amp`);
- 10k-step linear warmup → linear decay, gradient clipping at 1.0;
- per-rank streaming via `datasets.distributed.split_dataset_by_node`;
- per-step EMA loss, wandb config populated from the run variables, rank-0 full-state
  checkpoints saved locally.


In [ ]:
# Optional GPU mini-run: five real optimization steps on the toy batch.
# Verifies the full train step (autocast -> backward -> clip -> AdamW) end to end.
if torch.cuda.is_available():
    dev = torch.device("cuda")
    m = TransformerModel(vocab_size=tokenizer.vocab_size).to(dev)
    m.use_checkpoint = True
    m.train()
    opt = torch.optim.AdamW(m.parameters(), lr=1e-4, weight_decay=0.01)
    amp_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    scaler = torch.amp.GradScaler("cuda", enabled=(amp_dtype == torch.float16))
    b = {k: v.to(dev) for k, v in collate(samples).items()}
    for step in range(1, 6):
        opt.zero_grad(set_to_none=True)
        with torch.autocast(device_type="cuda", dtype=amp_dtype):
            loss, _ = m(b["input_ids"], attention_mask=b["attention_mask"], labels=b["labels"])
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0)
        scaler.step(opt); scaler.update()
        print(f"step {step}: loss {loss.item():.4f}")
else:
    print("No GPU in this session — run `torchrun --nproc_per_node=N src/train.py` on the GPU box.")


**Weights & Biases**

`train.py` calls `wandb.init` on rank 0 only. Authenticate once per machine with
`wandb login` or `export WANDB_API_KEY=...`.


**Benchmark plan** *(next step — no numbers claimed until this runs)*

Four configurations × {1, 3} GPUs, measuring median step time and peak memory
(`torch.cuda.max_memory_allocated`):

1. baseline fp32, no checkpointing
2. + AMP (bf16/fp16)
3. + activation checkpointing
4. + FlashAttention-2 (maskless path)

Results land in `docs/` and in the README's Status section when measured.
